# Batch: Study intensities at membrane

## Imports

In [1]:
from pathlib import Path
from cellpose_napari import CellPoseInference, ImageUtils
import xarray as xr
import tifffile as tiff
from skimage.measure import regionprops
import numpy as np
import shutil
import pandas as pd
from tqdm import tqdm
from skimage.morphology import dilation, erosion
from skimage.morphology.footprints import footprint_rectangle

## Settings

In [2]:
MEMBRANE_THICKNESS = 7
CHANNELS = [
    "DAPI",
    "GFP",
    "DSRED",
#    "FARRED"
]
CELLS_DIAMETER = 150
MIN_LABEL_SIZE = 500
SEGMENTATION_CHANNELS = ["DSRED"]
# MEASUREMENTS_CHANNELS = ["GFP", "DSRED","FARRED"]
MEASUREMENTS_CHANNELS = ["GFP", "DSRED"]

PIXEL_SIZE_XY = 0.1031746

ROOT_FOLDER = Path("/home/baecker/Documents/mri/in/2026/open-desk/2026-10-21/marion")
INPUT_FOLDER_NAME  = "test"
LABELS_FOLDER_NAME = "Labels"
OUTPUT_FOLDER_NAME = "histos"

## Checks

In [3]:
def do_path_checks():
    if not ROOT_FOLDER.exists():
        print(f"Root folder does not exist: {ROOT_FOLDER}")
        return
    p1 = ROOT_FOLDER / INPUT_FOLDER_NAME
    if not p1.exists():
        print(f"Input folder does not exist: {p1}")
        return
    p2 = ROOT_FOLDER / LABELS_FOLDER_NAME
    if p2.exists():
        shutil.rmtree(p2)
    p2.mkdir()
    p3 = ROOT_FOLDER / OUTPUT_FOLDER_NAME
    if p3.exists():
        shutil.rmtree(p3)
    p3.mkdir()
    print("All checks passed and necessary folders are prepared.")

do_path_checks()

All checks passed and necessary folders are prepared.


## Segmentation

In [4]:
images_folder = ROOT_FOLDER / INPUT_FOLDER_NAME
labels_folder = ROOT_FOLDER / LABELS_FOLDER_NAME
input_tiff_list = list(images_folder.glob("*.tif"))
images_axes = "YX"

c1_index = CHANNELS.index(SEGMENTATION_CHANNELS[0])
c2_index = CHANNELS.index(SEGMENTATION_CHANNELS[1]) if len(SEGMENTATION_CHANNELS) > 1 else None

for img_path in tqdm(input_tiff_list):
    img = tiff.imread(img_path)

    cells_channel = img[c1_index]
    cells_channel = xr.DataArray(cells_channel, dims=list(images_axes))
    cells_channel = ImageUtils.ensureAxes(cells_channel)

    nuclei_channel = img[c2_index] if c2_index is not None else None
    if nuclei_channel is not None:
        nuclei_channel = xr.DataArray(nuclei_channel, dims=list(images_axes))
        nuclei_channel = ImageUtils.ensureAxes(nuclei_channel)

    worker = CellPoseInference( # using default model
        ch_main=cells_channel,
        ch_secondary=nuclei_channel,
        diameter=CELLS_DIAMETER,
        anisotropy=1.0,
        min_size=MIN_LABEL_SIZE,
        cell_prob=0.0,
        flow_thr=0.4,
        flow_smooth=1,
        use_gpu=True,
        kill_border=True
    )

    for _ in worker.run():
        pass

    segmented_cells = worker.output_buffer

    if segmented_cells is None:
        print(f"Segmentation failed for {img_path.name}.")
        continue

    output_path = labels_folder / img_path.name
    tiff.imwrite(
        output_path, 
        segmented_cells,
        imagej=True
    )
print("Segmentation done!")

  0%|          | 0/3 [00:00<?, ?it/s]

Sanity check passed successfully
[GUI INFO] : WRITING LOG OUTPUT TO /home/baecker/.cellpose/run.log

cellpose version: 	4.2.1.1 
platform:       	linux 
python version: 	3.10.20 
torch version:  	2.12.1+cu130
2026-10-22 14:52:09,259 [io INFO] WRITING LOG OUTPUT TO /home/baecker/.cellpose/run.log
2026-10-22 14:52:09,260 [io INFO] 
cellpose version: 	4.2.1.1 
platform:       	linux 
python version: 	3.10.20 
torch version:  	2.12.1+cu130
2026-10-22 14:52:09,445 [core INFO] ** TORCH CUDA version installed and working. **
2026-10-22 14:52:09,445 [core INFO] >>>> using GPU (CUDA)
2026-10-22 14:52:10,877 [models INFO] >>>> loading model /home/baecker/.cellpose/models/cpsam



Processing time points:   0%|          | 0/1 [00:00<?, ?it/s]/home/baecker/miniforge3/envs/cellpose-napari/lib/python3.10/site-packages/cellpose/dynamics.py:541: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  coo = torch.sparse_coo_tensor(pt, torch.ones(pt.shape[1], device=pt.device, dtype=torch.int),

 33%|███▎      | 1/3 [00:14<00:28, 14.19s/it]

Sanity check passed successfully
[GUI INFO] : WRITING LOG OUTPUT TO /home/baecker/.cellpose/run.log

cellpose version: 	4.2.1.1 
platform:       	linux 
python version: 	3.10.20 
torch version:  	2.12.1+cu130
2026-10-22 14:52:23,448 [io INFO] WRITING LOG OUTPUT TO /home/baecker/.cellpose/run.log
2026-10-22 14:52:23,449 [io INFO] 
cellpose version: 	4.2.1.1 
platform:       	linux 
python version: 	3.10.20 
torch version:  	2.12.1+cu130
2026-10-22 14:52:23,449 [core INFO] ** TORCH CUDA version installed and working. **
2026-10-22 14:52:23,450 [core INFO] >>>> using GPU (CUDA)
2026-10-22 14:52:24,841 [models INFO] >>>> loading model /home/baecker/.cellpose/models/cpsam



 67%|██████▋   | 2/3 [00:24<00:12, 12.05s/it]

Sanity check passed successfully
[GUI INFO] : WRITING LOG OUTPUT TO /home/baecker/.cellpose/run.log

cellpose version: 	4.2.1.1 
platform:       	linux 
python version: 	3.10.20 
torch version:  	2.12.1+cu130
2026-10-22 14:52:33,996 [io INFO] WRITING LOG OUTPUT TO /home/baecker/.cellpose/run.log
2026-10-22 14:52:33,997 [io INFO] 
cellpose version: 	4.2.1.1 
platform:       	linux 
python version: 	3.10.20 
torch version:  	2.12.1+cu130
2026-10-22 14:52:33,997 [core INFO] ** TORCH CUDA version installed and working. **
2026-10-22 14:52:33,998 [core INFO] >>>> using GPU (CUDA)
2026-10-22 14:52:35,400 [models INFO] >>>> loading model /home/baecker/.cellpose/models/cpsam



100%|██████████| 3/3 [00:37<00:00, 12.52s/it]

Segmentation done!


## Measurements

In [5]:
UPPER_VALUE = 65536
NUM_BINS    = 128

In [11]:
output_folder = ROOT_FOLDER / OUTPUT_FOLDER_NAME

for img_path in tqdm(input_tiff_list):
    labels_path = labels_folder / img_path.name
    img = tiff.imread(img_path)
    labels = tiff.imread(labels_path)
    listOfLabels = {l for l in np.unique(labels) if l != 0}
    innerLabels = np.zeros_like(labels)
    membraneLabels = np.zeros_like(labels)
    results = []  

    for label in listOfLabels:
        mask    = (labels == label).astype(np.uint8)
        dilated = dilation(mask, footprint=footprint_rectangle((MEMBRANE_THICKNESS, MEMBRANE_THICKNESS)))
        eroded  = erosion(mask, footprint=footprint_rectangle((MEMBRANE_THICKNESS, MEMBRANE_THICKNESS)))

        innerMask    = eroded # erosion(eroded, footprint=footprint_rectangle((MEMBRANE_THICKNESS, MEMBRANE_THICKNESS)))
        membraneMask = dilated - eroded

        innerLabels = np.logical_or(innerLabels, (innerMask * label).astype(np.uint16))
        membraneLabels = np.logical_or(membraneLabels, (membraneMask * label).astype(np.uint16))

        if innerMask.sum() < 10 or membraneMask.sum() < 10:
            continue

        buffer = {
            'image': img_path.name,
            'label': label,
        }

        for meas_channel in MEASUREMENTS_CHANNELS:
            c_index = CHANNELS.index(meas_channel)
            meas_data = img[c_index]

            bin_edges = np.linspace(0, UPPER_VALUE, NUM_BINS + 1, endpoint=True)
            bin_cols = [
                f"[{int(bin_edges[i])}, {int(bin_edges[i+1])}]"
                for i in range(NUM_BINS)
            ]


            for region_name, region_mask in (("membrane", membraneMask), ("cyto", innerMask)):
                values = meas_data[region_mask.astype(bool)]
                values = values[(values >= 0) & (values < UPPER_VALUE)]

                hist, _ = np.histogram(values, bins=bin_edges)

                row = {
                    "image": img_path.name,
                    "label": label,
                    "channel": meas_channel,
                    "region": region_name,
                    "total": int(region_mask.astype(bool).sum()),
                }
                row.update({col: (int(count) / row['total']) * 100 for col, count in zip(bin_cols, hist)})
                results.append(row)

    df = pd.DataFrame(results)
    df.to_csv(output_folder / f"histos-{img_path.stem}.csv", index=False)

print("Measurement extraction done!")

100%|██████████| 3/3 [00:08<00:00,  2.99s/it]

Measurement extraction done!


## Control plot

In [8]:
import re
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# ---------------- Paramètres des plots de contrôle ----------------
PLOT_REGION      = "membrane"
PLOT_CHANNELS    = MEASUREMENTS_CHANNELS   # ordre de gauche à droite
PLOT_NORMALIZE   = True    # True : fraction des pixels de la cellule (cellules de tailles différentes comparables)
PLOT_COVERAGE    = 0.999   # n'affiche que les bins couvrant 99.9 % des pixels (coupe la traîne vide)
PLOT_MIN_BINS    = 10       # nombre minimal de bins affichés
PLOT_LOG_Y       = False   # True pour mieux voir les bins faiblement peuplés
PLOT_SHOW_POINTS = True    # superpose chaque cellule en petits points
PLOT_MAX_XTICKS  = 12
PLOT_SAVE        = True    # sauvegarde aussi les figures dans output_folder / "control_plots"
# ------------------------------------------------------------------

def bin_columns(df):
    return [c for c in df.columns if re.fullmatch(r"\[\s*\d+(\.\d+)?,\s*\d+(\.\d+)?\]", str(c))]

def bin_lower(col):
    return float(col.strip("[]").split(",")[0])

def last_useful_bin(counts, coverage, min_bins):
    """Index du dernier bin à afficher : couvre `coverage` des pixels de toutes les cellules réunies."""
    pooled = counts.sum(axis=0)
    if pooled.sum() == 0:
        return min_bins - 1
    cum = np.cumsum(pooled) / pooled.sum()
    return max(int(np.searchsorted(cum, coverage)), min_bins - 1)

def plot_control(df, image_name):
    bins = sorted(bin_columns(df), key=bin_lower)
    sub = df[df["region"] == PLOT_REGION]
    fig, axes = plt.subplots(1, len(PLOT_CHANNELS), figsize=(6 * len(PLOT_CHANNELS), 5),
                             sharey=PLOT_NORMALIZE and not PLOT_LOG_Y)

    for ax, ch in zip(np.atleast_1d(axes), PLOT_CHANNELS):
        g = sub[sub["channel"] == ch]
        if g.empty:
            ax.set_title(f"{ch} - {PLOT_REGION} (aucune donnée)")
            ax.axis("off")
            continue

        counts = g[bins].to_numpy(dtype=float)
        values = counts / g["total"].to_numpy(dtype=float)[:, None] if PLOT_NORMALIZE else counts

        last = last_useful_bin(counts, PLOT_COVERAGE, PLOT_MIN_BINS)
        shown = bins[: last + 1]
        hidden_frac = 1 - counts[:, : last + 1].sum() / counts.sum() if counts.sum() > 0 else 0

        long_df = pd.DataFrame(values[:, : last + 1], columns=shown).melt(var_name="bin", value_name="value")

        sns.boxplot(data=long_df, x="bin", y="value", order=shown, ax=ax, color="skyblue",
                    fliersize=0 if PLOT_SHOW_POINTS else 2, linewidth=1)
        if PLOT_SHOW_POINTS:
            sns.stripplot(data=long_df, x="bin", y="value", order=shown, ax=ax,
                          color="0.25", size=2, alpha=0.5, jitter=0.25)

        # Étiquettes lisibles : borne basse du bin, une étiquette sur k
        step = max(1, int(np.ceil(len(shown) / PLOT_MAX_XTICKS)))
        ticks = list(range(0, len(shown), step))
        ax.set_xticks(ticks)
        ax.set_xticklabels([f"{bin_lower(shown[i]):g}" for i in ticks], rotation=45, ha="right")

        if PLOT_LOG_Y:
            ax.set_yscale("symlog", linthresh=1e-3 if PLOT_NORMALIZE else 1)
        ax.set_xlabel("Intensité (borne basse du bin)")
        ax.set_ylabel("Fraction des pixels de la cellule" if PLOT_NORMALIZE else "Comptes par cellule")
        ax.set_title(f"{ch} - {PLOT_REGION}  (n = {len(g)} cellules)")
        note = f"{len(shown)}/{len(bins)} bins affichés"
        if hidden_frac > 0:
            note += f"\n{hidden_frac:.2%} des pixels hors zone"
        ax.text(0.98, 0.97, note, transform=ax.transAxes, ha="right", va="top", fontsize=8,
                bbox=dict(boxstyle="round", fc="white", ec="0.8"))

    fig.suptitle(f"Contrôle histogrammes {PLOT_REGION} (image : {image_name})")
    fig.tight_layout()
    return fig

csv_files = sorted(output_folder.glob("histos-*.csv"))
plot_dir = output_folder / "control_plots"
if PLOT_SAVE:
    plot_dir.mkdir(exist_ok=True)

for csv_path in csv_files:
    df = pd.read_csv(csv_path)
    if df.empty:
        print(f"Aucune cellule mesurée dans {csv_path.name}")
        continue
    fig = plot_control(df, df["image"].iloc[0])
    if PLOT_SAVE:
        fig.savefig(plot_dir / f"{csv_path.stem}_{PLOT_REGION}_boxplots.png", dpi=150, bbox_inches="tight")
    plt.show()

if PLOT_SAVE:
    print(f"Plots sauvegardés dans : {plot_dir}")


/tmp/ipykernel_50083/2073343412.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_50083/2073343412.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Plots sauvegardés dans : /home/baecker/Documents/mri/in/2026/open-desk/2026-10-21/marion/histos/control_plots


/tmp/ipykernel_50083/2073343412.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
